# Part C
**Members**: Daniel Andrade - Andrés Vega - Carlos Flores

## Quick Notes
### Classical scanner
`semgrep` is run against the original `labs/vuln-web/app/vulnweb_app.py` (semgrep ignores comments, so
there is nothing to strip) with three registry configs: `auto`, `p/security-audit` and `p/python`.
`run_semgrep()` re-runs semgrep and saves the raw JSON to [semgrep_output.json](./semgrep_output.json).

All three configs return **zero results and zero errors**. The likely reason is that registry rules are
written for frameworks (Flask, Django) and their taint sources, and this app is plain `http.server` +
`sqlite3`. **Caveat:** we have no positive control (a file these configs are known to flag), so we cannot
fully separate "the rules do not match this code" from "no rules ran".

### LLM reviewer
`qwen2.5:3b` through Ollama (`seclab.LLM`), temperature $0.7$, seeds $0$–$5$: six runs per configuration.

| config | prompt | source shown to the model | role |
|---|---|---|---|
| `v1-full` | v1 | full file, with the docstring and `# VULN` comments | baseline: what our first draft measured |
| `v1` | v1 | docstring and comments removed | measures how much the comments leaked |
| `v2` | v1 + *"Report only security vulnerabilities (OWASP Top 10 / CWE classes). Do not report code style, typing, naming, or other code-quality issues."* | docstring and comments removed | **main result** |

- **Why strip the source:** the file labels every bug, in comments (`# VULN: SQL injection...`) and in the
  module docstring ("reflected XSS sink", "the FIXED version of the XSS sink (a true negative)"). The
  `v1-full` answers copy those comments word for word ("The whole point of the SQLi lab"), so the model was
  reading the answer key.
- **Why v2:** v1 reported non-security findings (missing type hints). v2 adds a generic scope rule that names
  no specific finding. We wrote v2 *after* seeing the v1 output (see `honesty.md`).
- **Parsing:** `parse_llm_review()` maps each answer line to a `(rule, function)` pair. The rule normally
  comes from the vulnerability-type field. When the model leaves that field generic or just repeats the
  function name (`LOW do_ping in do_ping: ... command injection ...`), the rule is read from the reason. Every
  finding records which happened in the `mapped_by` column of `findings.csv`.

## Setup

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "seclab").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [2]:
from homework.hw2.andrade.duel.duel import SOURCE_PATH, run_llm_review, run_semgrep
from seclab.scan import ScanResult

full_source = SOURCE_PATH.read_text(encoding="utf-8")

file_contents = ""
with open(SOURCE_PATH, "r", encoding="utf-8") as file:
    for i, line in enumerate(file, start=1):
        # docstrings and comments should be ignored to avoid leaking information to the model
        if i <= 25 or "#" in line:
            continue
        file_contents += line

semgrep_results = run_semgrep(SOURCE_PATH)

SEEDS = range(6)
CONFIGS = {
    "v1-full": ("v1", full_source),    # baseline: docstring + comments included
    "v1":      ("v1", file_contents),  # same prompt, hints removed
    "v2":      ("v2", file_contents),  # + security-only scope rule (MAIN RESULT)
}

llm_runs: dict[str, list[list[ScanResult]]] = {}
llm_dropped: dict[str, list[list[str]]] = {}
for name, (prompt_version, source) in CONFIGS.items():
    llm_runs[name], llm_dropped[name] = [], []
    for seed in SEEDS:
        dropped: list[str] = []
        llm_runs[name].append(run_llm_review(source, seed=seed, temperature=0.7,
                                             prompt_version=prompt_version, dropped=dropped))
        llm_dropped[name].append(dropped)

runs = llm_runs["v2"]  # main result, used in sections 1-4

print(f"semgrep: {len(semgrep_results)} findings")
for name, rs in llm_runs.items():
    print(f"{name:<8} findings per seed: {[len(r) for r in rs]}")

semgrep: 0 findings
v1-full  findings per seed: [2, 3, 2, 3, 3, 3]
v1       findings per seed: [1, 1, 1, 1, 5, 2]
v2       findings per seed: [2, 2, 2, 2, 2, 2]


## 1. Precision, recall, F1

Main result: `semgrep` vs. the six `v2` runs, scored against
[`ground_truth.json`](../../duel-2-web/ground_truth.json) (3 real bugs, built by reading the source).

In [ ]:
from homework.hw2.andrade.duel.duel import load_ground_truth
from seclab.scan import compare_scanners, print_comparison

gt = load_ground_truth()

tools = {"semgrep": semgrep_results}
for i, run in enumerate(runs):
    tools[f"llm-v2-run{i}"] = run

print_comparison(compare_scanners(tools, gt))

  tool             TP   FP   FN    prec  recall     F1
  ----------------------------------------------------
  semgrep           0    0    3      0%      0%   0.00
  llm-v2-run0       1    1    2     50%     33%   0.40
  llm-v2-run1       1    1    2     50%     33%   0.40
  llm-v2-run2       2    0    1    100%     67%   0.80
  llm-v2-run3       2    0    1    100%     67%   0.80
  llm-v2-run4       1    1    2     50%     33%   0.40
  llm-v2-run5       1    1    2     50%     33%   0.40

  Read this like a security engineer, not a leaderboard:

  - HIGH RECALL, LOW PRECISION (finds everything, cries wolf): an LLM often lands
    here. Every false positive is analyst time; a tool that reports 40 bugs where
    6 are real will be ignored within a week.
  - HIGH PRECISION, LOW RECALL (quiet but misses things): a narrow ruleset often
    lands here. What it misses is what hurts you.
  - The interesting cell is the FALSE POSITIVES the LLM invented and the TRUE bugs
    the scanner MISSED

- **semgrep:** 0 TP, 0 FP, recall $0$.
- **LLM (v2):** precision $0.50$–$1.00$ (mean $0.67$), recall $0.33$–$0.67$ (mean $0.44$), mean F1 $0.53$.
  - SQL injection is the only bug it finds in every run.
  - Command injection is found in 2 of 6 runs (seeds 2 and 3); XSS is never found.
  - The LLM "wins" on recall only because semgrep reports nothing.

## 2. LLM's Best Hallucination & Scanner's Worst Miss

### LLM's best hallucination
> `[v2, seed 5]  LOW do_ping in do_ping: SQL query may be vulnerable to SQL injection if host is directly used in the query.`

`do_ping` contains no SQL at all; the real bug there is **command injection**. The model saw "user input
concatenated into a string" and reused its `do_login` explanation. It found the right place but named the
wrong bug, so the run scores a false positive (`sqli`, `do_ping`) *and* a miss (`cmdi`, `do_ping`). The same
seed under prompt v1 makes the same claim, so this is not a parser artefact.

The other false positives, all printed below:
- **v2, seed 1:** *Security Misconfiguration in `main`*: "running on a public port (8000)... could be exploited
  for command injection". Listening on a port does not enable command injection; the causal chain is invented.
- **v1, seed 4:** four *Missing type hint* findings in `do_login` / `do_ping`. Both functions are fully annotated
  (`def do_login(user: str, pw: str) -> str`), so the model reports a bug in code that is not there. These are
  also not security claims, which is why we wrote prompt v2.
- **Debatable (v2, seeds 0 and 4):** *hardcoded secret* in `make_db` and *missing Content-Security-Policy* in
  `Handler`.
  - Both are literally true: `CANARY_SECRET` is hard-coded, and no CSP header is sent.
  - Neither is a vulnerability in our ground truth: the secret is the lab's deliberate canary, and a missing CSP is
    absent hardening, not an exploitable bug.
  - We count them as false positives. Excluding them would raise mean v2 precision from $0.67$ to $0.83$.
- **Not scored, but a hallucinated fix (v1-full, seed 0):** "Fixed version would be to use `os.system(command)`".
  `os.system` is the vulnerable sink itself.

### Scanner's worst miss
The **SQL injection in `do_login`**: a textbook f-string query, and the one bug every LLM run found. All
three misses share one cause: no registry rule matched this stdlib `http.server` + `sqlite3` code. Because we
have no positive control, we cannot rule out that the configs loaded no applicable rules at all.

In [4]:
# Evidence for section 2: every LLM false positive (with the raw line), and semgrep's misses.
from seclab.scan import classify

def false_positives(config):
    rows = []
    for seed, run in enumerate(llm_runs[config]):
        fps = set(classify(run, gt)["false_positives"])
        rows += [(seed, r) for r in run if r.key() in fps]
    return rows

for config in ("v2", "v1", "v1-full"):
    fps = false_positives(config)
    print(f"== {config}: {len(fps)} false positive(s)")
    for seed, r in fps:
        print(f"  seed {seed}: {r.key()}  [{r.confidence}]\n      {r.raw}")

print("\nsemgrep missed (real bugs not reported):", classify(semgrep_results, gt)["missed"])

== v2: 4 false positive(s)
  seed 0: ('hardcoded-secret', 'make_db')  [model:prose]
      [LOW] Code Review: The `make_db` function uses a hardcoded secret (`CANARY_SECRET`) which could be a security risk if the secret is not changed.
  seed 1: ('security-misconfiguration', 'main')  [model:type]
      [LOW] Security Misconfiguration in main: The function is running on a public port (8000) which is not secure. This could be exploited for command injection or other attacks.
  seed 4: ('missing-content-security-policy', 'handler')  [model:slug]
      [LOW] Missing Content Security Policy in Handler: The server does not enforce a Content Security Policy, which can lead to various security risks like XSS and script injection.
  seed 5: ('sqli', 'do_ping')  [model:reason]
      LOW do_ping in do_ping: SQL query may be vulnerable to SQL injection if host is directly used in the query.
== v1: 5 false positive(s)
  seed 4: ('missing-type-hint-for-function-parameters', 'do_login')  [model:slug]


## 3. Non-determinism

The same model, prompt and temperature ($0.7$), run with seeds $0$–$5$. For each configuration we report
which findings are **stable** (present in every run) and which **flicker**. We also show the per-seed
precision and recall, the severity given to the same bug, and answers that broke the requested format.

In [5]:
from collections import Counter

def stability(config):
    key_sets = [{r.key() for r in run} for run in llm_runs[config]]
    appear = Counter(k for ks in key_sets for k in ks)
    n = len(key_sets)
    print(f"== {config}")
    print(f"{'finding':<52}{'runs':>6}  state")
    print("-" * 70)
    for k in sorted(appear, key=lambda k: (-appear[k], k)):
        state = "STABLE" if appear[k] == n else "flaky"
        print(f"{str(k):<52}{appear[k]:>3}/{n}  {state:<6} ({'real' if k in gt else 'FP'})")

    comp = compare_scanners({f"s{i}": run for i, run in enumerate(llm_runs[config])}, gt)
    print("per-seed precision:", [round(d["precision"], 2) for d in comp.values()])
    print("per-seed recall:   ", [round(d["recall"], 2) for d in comp.values()])
    sev = [next((r.severity for r in run if r.key() == ("sqli", "do_login")), "-")
           for run in llm_runs[config]]
    print("severity of ('sqli', 'do_login') per seed:", sev)
    print("findings whose rule was read from the reason, per seed:",
          [sum(r.confidence == "model:reason" for r in run) for run in llm_runs[config]])
    print("off-format / contradictory lines:")
    for seed, d in enumerate(llm_dropped[config]):
        if d:
            print(f"  seed {seed}: {d}")
    print()

for config in ("v2", "v1"):
    stability(config)

== v2
finding                                               runs  state
----------------------------------------------------------------------
('sqli', 'do_login')                                  6/6  STABLE (real)
('cmdi', 'do_ping')                                   2/6  flaky  (real)
('hardcoded-secret', 'make_db')                       1/6  flaky  (FP)
('missing-content-security-policy', 'handler')        1/6  flaky  (FP)
('security-misconfiguration', 'main')                 1/6  flaky  (FP)
('sqli', 'do_ping')                                   1/6  flaky  (FP)
per-seed precision: [0.5, 0.5, 1.0, 1.0, 0.5, 0.5]
per-seed recall:    [0.33, 0.33, 0.67, 0.67, 0.33, 0.33]
severity of ('sqli', 'do_login') per seed: ['high', 'high', 'low', 'low', 'medium', 'low']
findings whose rule was read from the reason, per seed: [0, 1, 2, 2, 0, 2]
off-format / contradictory lines:
  seed 4: ['[NONE]  <- NONE after 2 finding(s)']

== v1
finding                                               runs  sta

- **Same seed, same answer.** Re-running a seed reproduces its answer exactly. The LLM cache was cleared
  during the homework, and the regenerated `v1-full` and `v1` answers were identical to the ones saved
  earlier. A "run it twice" check with a fixed seed would therefore show zero variance; the variance we
  measure comes from sampling across seeds.
- **Across seeds (v2):**
  - Only `('sqli', 'do_login')` is stable (6/6). Command injection appears in 2/6 runs.
  - Each false positive appears in a single run: four different false positives in four runs.
  - Precision moves between $0.5$ and $1.0$ and recall between $0.33$ and $0.67$.
  - On any single run the model may miss a real bug *and* invent a different one.
- **Severity is not reproducible either.** The same SQL injection is rated HIGH, HIGH, LOW, LOW, MEDIUM, LOW
  across the six v2 seeds.
- **Format drift:**
  - In 4 of 6 v2 runs the vulnerability-type field is the function name or a generic word, so the class had
    to be read from the reason.
  - v2 seed 4 answers `[NONE]` *after* listing findings.
  - Under v1, every one of the six runs ends with `NONE` after its findings.

A single LLM scan is therefore not reproducible evidence: the result depends on the seed, not only on the
code.

## 4. Access control

**Did either tool find an access-control issue?** No. That is the correct result: no tool, configuration or
seed reported `broken-access-control`.

In [6]:
reported = ({r.key() for rs in llm_runs.values() for run in rs for r in run}
            | {r.key() for r in semgrep_results})
bac = sorted(k for k in reported if k[0] == "broken-access-control")
print("broken-access-control reported by any tool / config / seed?", bool(bac), bac)

broken-access-control reported by any tool / config / seed? False []


`vuln-web` has no authentication, session or role logic, so there is no access-control decision to break.
The ground truth lists `broken-access-control` as *not present*. Neither tool hallucinated it, which is a good
outcome. It is also a reminder that Broken Access Control (OWASP 2021 #1) is about application *intent*, not
syntax: a pattern-based scanner and a code-reading LLM are both poorly placed to find it, even when it is real.

## 5. What the comments and the prompt changed

Mean over the six seeds per configuration, plus how many seeds found each real bug.

In [7]:
bugs = sorted(gt)
print(f"{'config':<9}{'mean P':>8}{'mean R':>8}{'mean F1':>9}{'FPs':>5}   "
      + "  ".join(f"{rule}@{loc}" for rule, loc in bugs))
for config, rs in llm_runs.items():
    comp = compare_scanners({f"s{i}": run for i, run in enumerate(rs)}, gt).values()
    n = len(rs)
    mean = lambda m: sum(d[m] for d in comp) / n
    fps = sum(d["fp"] for d in comp)
    found = [sum(b in {r.key() for r in run} for run in rs) for b in bugs]
    print(f"{config:<9}{mean('precision'):>8.2f}{mean('recall'):>8.2f}{mean('f1'):>9.2f}{fps:>5}   "
          + "  ".join(f"{f}/{n}".center(len(f'{r}@{l}')) for f, (r, l) in zip(found, bugs)))

config     mean P  mean R  mean F1  FPs   cmdi@do_ping  sqli@do_login  xss@do_reflect
v1-full      1.00    0.89     0.93    0       6/6            6/6            4/6      
v1           0.78    0.33     0.44    5       0/6            6/6            0/6      
v2           0.67    0.44     0.53    4       2/6            6/6            0/6      


- **The comments were the answer key.** With them (`v1-full`), mean recall is $0.89$ with zero false positives.
  Without them, under the same prompt (`v1`), mean recall drops to $0.33$: command injection and XSS are never
  found. Our first draft's "perfect precision, high recall" was mostly the model reading `# VULN` comments.
- **v2 removed the type-hint noise, but not the false positives.** It produced 4 false positives instead of 5,
  spread over four runs instead of two, so mean precision is *lower* ($0.67$ vs. $0.78$). It recovered command
  injection in two seeds, raising mean recall to $0.44$.
- **Six seeds are too few** to claim v2 is better than v1. The difference is within the seed-to-seed spread
  shown in section 3.

## 6. Result CSVs

Written beside this notebook, for submission:
- `findings.csv`: every finding of every tool, configuration and seed, with `mapped_by` and `in_ground_truth`.
- `metrics.csv`: TP / FP / FN / precision / recall / F1 per tool, configuration and seed.
- `dropped_lines.csv`: every answer line that was not parsed as a finding, so nothing is silently discarded.

In [8]:
import csv

DUEL_DIR = ROOT / "homework" / "hw2" / "andrade" / "duel"

with open(DUEL_DIR / "findings.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["config", "tool", "seed", "rule", "location", "severity", "mapped_by",
                "in_ground_truth", "raw"])
    for r in semgrep_results:
        w.writerow(["-", "semgrep", "-", r.rule, r.location, r.severity, r.confidence,
                    r.key() in gt, r.raw])
    for config, rs in llm_runs.items():
        for seed, run in enumerate(rs):
            for r in run:
                w.writerow([config, "llm", seed, r.rule, r.location, r.severity, r.confidence,
                            r.key() in gt, r.raw])

scored = {("-", "semgrep", "-"): semgrep_results}
for config, rs in llm_runs.items():
    for seed, run in enumerate(rs):
        scored[(config, "llm", seed)] = run

with open(DUEL_DIR / "metrics.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["config", "tool", "seed", "tp", "fp", "fn", "precision", "recall", "f1"])
    for (config, tool, seed), found in scored.items():
        d = compare_scanners({"t": found}, gt)["t"]
        w.writerow([config, tool, seed, d["tp"], d["fp"], d["fn"],
                    f"{d['precision']:.2f}", f"{d['recall']:.2f}", f"{d['f1']:.2f}"])

with open(DUEL_DIR / "dropped_lines.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["config", "seed", "line"])
    for config, per_seed in llm_dropped.items():
        for seed, lines in enumerate(per_seed):
            for line in lines:
                w.writerow([config, seed, line])

print("wrote findings.csv, metrics.csv and dropped_lines.csv to", DUEL_DIR)

wrote findings.csv, metrics.csv and dropped_lines.csv to /Users/danielandrade/Documents/usfq/9th-semester/computer-security/cmp-5006E-fall26/homework/hw2/andrade/duel
